# Retail E-commerce Sales Analytics Pipeline
## Phase 2: Silver Stage 1 — Cleaning, Casting, Deduplication, Quarantine

This notebook covers:
- Orders: timestamp conversion, cleaning `unit_price`, identifying bad records, deduplication
- Customers & Products: date parsing, null handling, identifying bad records, deduplication
- Stores: deduplication
- Quarantine tables for every bad/rejected record

In [0]:
CATALOG = "retail_demo"  
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {RAW_SCHEMA}")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")

print("Using catalog:", CATALOG)

Using catalog: retail_demo


## 1. Orders — Clean, Cast, Deduplicate, Quarantine

### 1a. Combine batch + incremental orders
Both `bronze_orders` (historical) and `bronze_orders_incremental` (daily feed)
need to go through the same cleaning logic, so we union them first.
`coupon_code` only exists in the incremental table (Day-3 schema change) — we
add it as NULL to the batch table so the union works.

In [0]:
bronze_orders = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders")
bronze_orders_incr = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental")

# Add missing columns so both DataFrames have the same schema before union
if "coupon_code" not in bronze_orders.columns:
    bronze_orders = bronze_orders.withColumn("coupon_code", F.lit(None).cast("string"))
if "ingest_date" not in bronze_orders.columns:
    bronze_orders = bronze_orders.withColumn("ingest_date", F.lit(None).cast("string"))

common_cols = ["order_id", "order_ts", "customer_id", "product_id", "store_id",
               "quantity", "unit_price", "discount_pct", "gross_amount",
               "payment_method", "order_status", "coupon_code", "ingest_date",
               "source_file", "ingestion_ts", "load_type"]

all_orders_raw = (
    bronze_orders.select(common_cols)
    .unionByName(bronze_orders_incr.select(common_cols))
)

print("Total raw orders (batch + incremental):", all_orders_raw.count())

Total raw orders (batch + incremental): 18315


### 1b. Clean & cast
- `unit_price`: strip any non-numeric characters (e.g. "unknown", currency symbols) using regex, then cast to double
- `order_ts`: cast to timestamp
- `quantity`, `discount_pct`, `gross_amount`: cast to proper numeric types

In [0]:
orders_typed = (
    all_orders_raw
    .withColumn("order_ts_parsed", F.expr("try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss')"))
    .withColumn(
        "unit_price_clean",
        F.regexp_extract(F.col("unit_price"), r"(\d+\.?\d*)", 1)
    )
    .withColumn(
        "unit_price_parsed",
        F.when(F.col("unit_price_clean") == "", None)
         .otherwise(F.col("unit_price_clean").cast("double"))
    )
    .withColumn("quantity_parsed", F.expr("try_cast(quantity as int)"))
    .withColumn("discount_pct_parsed", F.expr("try_cast(discount_pct as double)"))
    .withColumn("gross_amount_parsed", F.expr("try_cast(gross_amount as double)"))
)

display(orders_typed.select(
    "order_id", "order_ts", "order_ts_parsed",
    "unit_price", "unit_price_parsed",
    "quantity", "quantity_parsed"
).limit(10))

order_id,order_ts,order_ts_parsed,unit_price,unit_price_parsed,quantity,quantity_parsed
O0000001,2025-12-15 04:49:00,2025-12-15T04:49:00.000Z,30260.29,30260.29,4,4
O0000002,2026-03-13 03:29:00,2026-03-13T03:29:00.000Z,30049.26,30049.26,4,4
O0000003,2026-03-05 07:20:00,2026-03-05T07:20:00.000Z,59685.88,59685.88,5,5
O0000004,2025-12-28 13:56:00,2025-12-28T13:56:00.000Z,18705.7,18705.7,1,1
O0000005,2025-12-31 17:43:00,2025-12-31T17:43:00.000Z,46727.14,46727.14,2,2
O0000006,2026-03-11 02:59:00,2026-03-11T02:59:00.000Z,12387.77,12387.77,2,2
O0000007,2025-12-26 03:16:00,2025-12-26T03:16:00.000Z,54612.57,54612.57,2,2
O0000008,2025-12-09 20:57:00,2025-12-09T20:57:00.000Z,29725.2,29725.2,1,1
O0000009,2026-01-20 19:25:00,2026-01-20T19:25:00.000Z,84092.97,84092.97,4,4
O0000010,2026-03-21 06:52:00,2026-03-21T06:52:00.000Z,31466.94,31466.94,5,5


### 1c. Identify bad records → Quarantine
A row is "bad" if any essential field is null/unparseable: `order_id`,
`customer_id`, `product_id`, `order_ts_parsed`, or `unit_price_parsed`.

In [0]:
is_bad_order = (
    F.col("order_id").isNull() |
    F.col("customer_id").isNull() |
    F.col("product_id").isNull() |
    F.col("order_ts_parsed").isNull() |
    F.col("unit_price_parsed").isNull()
)

orders_quarantine = orders_typed.filter(is_bad_order)
orders_ok = orders_typed.filter(~is_bad_order)

print("Bad/quarantined order rows:", orders_quarantine.count())
print("Good order rows (before dedup):", orders_ok.count())

(orders_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.quarantine_orders"))

Bad/quarantined order rows: 660
Good order rows (before dedup): 17655


### 1d. Deduplicate
A simple `.dropDuplicates()` isn't enough — if the same `order_id` was
re-sent (e.g. corrected), we want the **latest** version, not just any one.
We partition by `order_id`, order by `ingestion_ts` descending, and keep
only the top row per order.

In [0]:
order_window = Window.partitionBy("order_id").orderBy(F.col("ingestion_ts").desc())

silver1_orders_clean = (
    orders_ok
    .withColumn("row_num", F.row_number().over(order_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "unit_price_clean")
    .withColumnRenamed("order_ts_parsed", "order_ts_final")
    .withColumnRenamed("unit_price_parsed", "unit_price_final")
    .withColumnRenamed("quantity_parsed", "quantity_final")
    .withColumnRenamed("discount_pct_parsed", "discount_pct_final")
    .withColumnRenamed("gross_amount_parsed", "gross_amount_final")
)

(silver1_orders_clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean"))

print("silver1_orders_clean row count:", spark.table(f'{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean').count())

silver1_orders_clean row count: 17344


## 2. Customers — Clean, Cast, Deduplicate, Quarantine

In [0]:
bronze_customers = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers")

customers_typed = (
    bronze_customers
    .withColumn("signup_date_parsed", F.expr("try_to_date(signup_date, 'yyyy-MM-dd')"))
    .withColumn(
        "city_clean",
        F.when(F.col("city").isNull() | (F.trim(F.col("city")) == ""), F.lit("Unknown"))
         .otherwise(F.col("city"))
    )
    .withColumn(
        "segment_clean",
        F.when(F.col("segment").isNull() | (F.trim(F.col("segment")) == ""), F.lit("Unknown"))
         .otherwise(F.col("segment"))
    )
)

is_bad_customer = (
    F.col("customer_id").isNull() |
    F.col("customer_name").isNull()
)

customers_quarantine = customers_typed.filter(is_bad_customer)
customers_ok = customers_typed.filter(~is_bad_customer)

print("Bad/quarantined customer rows:", customers_quarantine.count())
print("Good customer rows (before dedup):", customers_ok.count())

(customers_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.quarantine_customers"))

Bad/quarantined customer rows: 0
Good customer rows (before dedup): 2560


In [0]:
customer_window = Window.partitionBy("customer_id").orderBy(F.col("ingestion_ts").desc())

silver1_customers_clean = (
    customers_ok
    .withColumn("row_num", F.row_number().over(customer_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "city", "segment")
    .withColumnRenamed("city_clean", "city")
    .withColumnRenamed("segment_clean", "segment")
    .withColumnRenamed("signup_date_parsed", "signup_date_final")
)

(silver1_customers_clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean"))

print("silver1_customers_clean row count:", spark.table(f'{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean').count())

silver1_customers_clean row count: 2500


## 3. Products — Clean, Cast, Deduplicate, Quarantine

In [0]:
bronze_products = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_products")

products_typed = (
    bronze_products
    .withColumn("created_date_parsed", F.to_date("created_date", "yyyy-MM-dd"))
    .withColumn(
        "category_clean",
        F.when(F.col("category").isNull() | (F.trim(F.col("category")) == ""), F.lit("Unknown"))
         .otherwise(F.col("category"))
    )
    .withColumn(
        "unit_price_clean",
        F.regexp_extract(F.col("unit_price"), r"(\d+\.?\d*)", 1)
    )
    .withColumn(
        "unit_price_parsed",
        F.when(F.col("unit_price_clean") == "", None)
         .otherwise(F.col("unit_price_clean").cast("double"))
    )
)

is_bad_product = (
    F.col("product_id").isNull() |
    F.col("product_name").isNull()
)

products_quarantine = products_typed.filter(is_bad_product)
products_ok = products_typed.filter(~is_bad_product)

print("Bad/quarantined product rows:", products_quarantine.count())
print("Good product rows (before dedup):", products_ok.count())

(products_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.quarantine_products"))

Bad/quarantined product rows: 0
Good product rows (before dedup): 830


In [0]:
product_window = Window.partitionBy("product_id").orderBy(F.col("ingestion_ts").desc())

silver1_products_clean = (
    products_ok
    .withColumn("row_num", F.row_number().over(product_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "category", "unit_price", "unit_price_clean")
    .withColumnRenamed("category_clean", "category")
    .withColumnRenamed("unit_price_parsed", "unit_price_final")
    .withColumnRenamed("created_date_parsed", "created_date_final")
)

(silver1_products_clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean"))

print("silver1_products_clean row count:", spark.table(f'{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean').count())

silver1_products_clean row count: 800


## 4. Stores — Deduplicate

In [0]:
bronze_stores = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_stores")

store_window = Window.partitionBy("store_id").orderBy(F.col("ingestion_ts").desc())

dim_store = (
    bronze_stores
    .withColumn("row_num", F.row_number().over(store_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("store_sk", F.row_number().over(Window.orderBy("store_id")))   # <-- NEW LINE
)

(dim_store.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.dim_store"))

print("dim_store row count:", spark.table(f'{CATALOG}.{SILVER_SCHEMA}.dim_store').count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_store row count: 75


## 5. Verification — Silver Stage 1 Summary

In [0]:
silver1_tables = [
    "silver1_orders_clean", "quarantine_orders",
    "silver1_customers_clean", "quarantine_customers",
    "silver1_products_clean", "quarantine_products",
    "dim_store",
]

for t in silver1_tables:
    full_name = f"{CATALOG}.{SILVER_SCHEMA}.{t}"
    cnt = spark.table(full_name).count()
    print(f"{t:30s} -> {cnt} rows")

silver1_orders_clean           -> 17344 rows
quarantine_orders              -> 660 rows
silver1_customers_clean        -> 2500 rows
quarantine_customers           -> 0 rows
silver1_products_clean         -> 800 rows
quarantine_products            -> 0 rows
dim_store                      -> 75 rows


## Summary — Phase 2 (Silver Stage 1)
- **Orders:** Combined batch + incremental orders, cleaned `unit_price`
  (regex-extracted digits, so "unknown" style values became NULL), parsed
  `order_ts` to a real timestamp, identified rows with null essential fields
  and quarantined them, then deduplicated on `order_id` keeping the latest
  `ingestion_ts` per order.
- **Customers:** Parsed `signup_date` (invalid formats like `31-31-2025`
  become NULL), replaced null `city`/`segment` with `"Unknown"`, quarantined
  rows missing `customer_id`/`customer_name`, deduplicated on `customer_id`.
- **Products:** Same pattern — cleaned `unit_price` ("unknown" → NULL),
  replaced null `category` with `"Unknown"`, quarantined bad rows,
  deduplicated on `product_id`.
- **Stores:** Deduplicated on `store_id` to produce `dim_store`.
- Every quarantined record is preserved in its own `quarantine_*` table for
  further investigation — nothing is silently dropped.